# CASCADE — Tier 0 · Trunk (GCN + GRU) + DeepHit head  (v2)

**Heavy-training half of the Colab bridge.** Run on a **GPU runtime**
(*Runtime → Change runtime type → T4 GPU*).

### Upload (same bundle as before)
From local `data/processed/`: **`train_bundle.npz`** and **`train_bundle_meta.json`**.

### v2 changes (to beat the XGBoost baseline on C-index)
- **Risk = expected bin index** (outlier-immune) instead of expected-minutes — the previous
  risk score was dominated by the 201,790-min tail bin.
- **Model selection on validation C-index** (was val NLL) — aligned with the reported metric.
- **Ranking loss weight 0.1 → 0.5**; trivial priority aux down-weighted 0.2 → 0.05.
- **ReduceLROnPlateau + dropout + grad-clip + more epochs** for convergence under temporal shift.

### Download back into local `models/`
`trunk_deephit.pt` (weights+config+cuts) and `preds_all.npz` (per-event predictions → conformal + allocator).


In [ ]:
# 0 · setup  (GPU runtime recommended)
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
pip('scikit-survival')   # concordance + integrated Brier — same yardstick as the local baseline

import os, json, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from sksurv.metrics import concordance_index_censored
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| device:', device)

In [ ]:
# 1 · load the bundle  (upload train_bundle.npz + train_bundle_meta.json if not present)
BUNDLE, METAF = 'train_bundle.npz', 'train_bundle_meta.json'
if not (os.path.exists(BUNDLE) and os.path.exists(METAF)):
    from google.colab import files
    print('Select train_bundle.npz AND train_bundle_meta.json:')
    files.upload()
z = np.load(BUNDLE, allow_pickle=True)
meta = json.load(open(METAF))
print('arrays:', {k: tuple(z[k].shape) for k in z.files})
N, V, K, L = meta['n_rows'], meta['num_nodes'], meta['n_bins'], meta['hist_len']
cf = meta['censored_frac']
print('rows N=%d  nodes V=%d  bins K=%d  history L=%d  censored=%.3f' % (N, V, K, L, cf))

In [ ]:
# 2 · tensors + symmetric-normalized adjacency  D^-1/2 (A+I) D^-1/2  (Kipf-Welling; no PyG needed)
def T(name, dtype): return torch.as_tensor(z[name], dtype=dtype).to(device)
X_num   = T('X_num',   torch.float32)
X_cat   = T('X_cat',   torch.long)
node_id = T('node_id', torch.long)
hist_idx= T('hist_idx',torch.long)
node_feat = T('node_feat', torch.float32)
edge_index= T('edge_index', torch.long)
bin_idx = T('bin_idx', torch.long)
road    = T('road_closure', torch.long)
prio    = T('priority_high', torch.long)
durations = z['durations'].astype('float32')
events    = z['events'].astype('int64')
cuts      = z['cuts'].astype('float32')
split     = z['split']

def norm_adj(ei, V):
    loops = torch.arange(V, device=ei.device)
    ei = torch.cat([ei, torch.stack([loops, loops])], dim=1)        # add self-loops
    src, dst = ei[0], ei[1]
    deg = torch.zeros(V, device=ei.device).scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))
    dinv = deg.pow(-0.5); dinv[torch.isinf(dinv)] = 0.0
    w = dinv[src] * dinv[dst]
    return torch.sparse_coo_tensor(ei, w, (V, V)).coalesce()

A = norm_adj(edge_index, V)
tr_idx = torch.where(torch.as_tensor(split == 0))[0].to(device)
va_idx = torch.where(torch.as_tensor(split == 1))[0].to(device)
te_idx = torch.where(torch.as_tensor(split == 2))[0].to(device)
ev_t = torch.as_tensor(events, dtype=torch.float32, device=device)
va_t, va_e = durations[split == 1], events[split == 1]   # for val C-index (same order as va_idx)
print('train/val/test:', tr_idx.numel(), va_idx.numel(), te_idx.numel())

In [ ]:
# 3 · model:  shared trunk (GCN spatial + GRU temporal) -> DeepHit multi-task head
class GCN(nn.Module):
    def __init__(self, fin, h, out):
        super().__init__(); self.l1 = nn.Linear(fin, h); self.l2 = nn.Linear(h, out)
    def forward(self, x, A):
        x = F.relu(torch.sparse.mm(A, self.l1(x)))
        return torch.sparse.mm(A, self.l2(x))

class EventFeat(nn.Module):
    # numeric features + categorical embeddings -> per-event vector (shared by GRU history bank)
    def __init__(self, n_num, cards, emb=8, out=64):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, emb) for c in cards])
        self.proj = nn.Linear(n_num + emb * len(cards), out)
    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.proj(torch.cat([xn] + e, dim=1))

class Trunk(nn.Module):
    def __init__(self, n_num, cards, fnode, d=64, p=0.2):
        super().__init__()
        self.d = d
        self.feat = EventFeat(n_num, cards, out=d)
        self.gcn  = GCN(fnode, 64, d)
        self.gru  = nn.GRU(d, d, batch_first=True)
        self.fuse = nn.Sequential(nn.Linear(d * 3, d), nn.ReLU(), nn.Dropout(p), nn.Linear(d, d))
    def event_vectors(self, xn, xc):
        return self.feat(xn, xc)                                  # [N, d]
    def forward(self, idx, all_vec):
        e = all_vec[idx]                                          # [B, d]  (event's own vector)
        g = self.gcn(node_feat, A)[node_id[idx]]                  # [B, d]  spatial road-graph context
        bank = torch.cat([all_vec, torch.zeros(1, self.d, device=all_vec.device)], 0)  # -1 -> zeros
        seq = bank[hist_idx[idx]]                                 # [B, L, d]  recent history at node
        _, hn = self.gru(seq)                                     # temporal
        return self.fuse(torch.cat([e, g, hn.squeeze(0)], dim=1)) # [B, d]

class Head(nn.Module):
    def __init__(self, d, K):
        super().__init__(); self.dur = nn.Linear(d, K); self.clo = nn.Linear(d, 1); self.pri = nn.Linear(d, 1)
    def forward(self, H):
        return self.dur(H), self.clo(H).squeeze(-1), self.pri(H).squeeze(-1)

In [ ]:
# 4 · DeepHit loss  =  censoring-aware NLL  +  ranking ; and the OUTLIER-IMMUNE risk score
def deephit_nll(logp, bins, ev):
    pmf = logp.exp()
    idx = bins.clamp(0, pmf.shape[1] - 1)[:, None]
    log_p_bin = logp.gather(1, idx).squeeze(1)                          # log P(event in its bin)
    tail = torch.flip(torch.cumsum(torch.flip(pmf, [1]), 1), [1])       # sum_{j>=k} pmf
    log_surv = tail.gather(1, idx).squeeze(1).clamp_min(1e-8).log()     # log P(survive past bin)
    return -(ev * log_p_bin + (1 - ev) * log_surv).mean()

def rank_loss(pmf, bins, ev, sigma=0.1, npairs=4096):
    cif = torch.cumsum(pmf, 1); dev = ev.bool()
    if dev.sum() < 2: return pmf.sum() * 0.0
    B = pmf.shape[0]
    ii = torch.randint(0, B, (npairs,), device=pmf.device); jj = torch.randint(0, B, (npairs,), device=pmf.device)
    valid = dev[ii] & (bins[ii] < bins[jj])
    if valid.sum() == 0: return pmf.sum() * 0.0
    ii, jj = ii[valid], jj[valid]; ti = bins[ii].clamp(0, pmf.shape[1] - 1)
    return torch.exp(-(cif[ii, ti] - cif[jj, ti]) / sigma).mean()

_BINS = np.arange(K, dtype=np.float32)
def risk_from_pmf(pmf):
    # higher risk = mass on EARLIER bins = shorter duration; expected bin index is outlier-immune
    return -(pmf * _BINS).sum(1)

In [ ]:
# 5 · build model + predict() helper
cards = meta['cat_cardinalities']; n_num = X_num.shape[1]; fnode = node_feat.shape[1]
trunk = Trunk(n_num, cards, fnode, d=64, p=0.2).to(device); head = Head(64, K).to(device)
opt = torch.optim.Adam(list(trunk.parameters()) + list(head.parameters()), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8)

@torch.no_grad()
def predict(idx):
    trunk.eval(); head.eval()
    all_vec = trunk.event_vectors(X_num, X_cat)
    dlog, clog, plog = head(trunk(idx, all_vec))
    return (F.softmax(dlog, 1).cpu().numpy(),
            torch.sigmoid(clog).cpu().numpy(), torch.sigmoid(plog).cpu().numpy())

def val_cindex():
    pmf, _, _ = predict(va_idx)
    return concordance_index_censored(va_e.astype(bool), va_t, risk_from_pmf(pmf))[0]

In [ ]:
# 6 · train  (select best epoch by VAL C-INDEX; LR schedule + grad-clip)
params = list(trunk.parameters()) + list(head.parameters())
def loss_on(idx):
    all_vec = trunk.event_vectors(X_num, X_cat)        # grad-tracked, shared by event + history
    dlog, clog, plog = head(trunk(idx, all_vec))
    logp = F.log_softmax(dlog, 1)
    nll = deephit_nll(logp, bin_idx[idx], ev_t[idx])
    rk  = rank_loss(logp.exp(), bin_idx[idx], ev_t[idx])
    clo = F.binary_cross_entropy_with_logits(clog, road[idx].float())
    pm  = prio[idx] >= 0
    pri = F.binary_cross_entropy_with_logits(plog[pm], prio[idx][pm].float()) if pm.any() else nll * 0
    return nll + 0.5 * rk + 0.2 * clo + 0.05 * pri, nll

bs, best_c, best_state, patience = 1024, -1.0, None, 0
for epoch in range(200):
    trunk.train(); head.train()
    perm = tr_idx[torch.randperm(tr_idx.numel(), device=device)]
    for s in range(0, perm.numel(), bs):
        opt.zero_grad(); loss, _ = loss_on(perm[s:s + bs]); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 5.0); opt.step()
    vc = val_cindex(); sched.step(vc)
    if vc > best_c + 1e-4:
        best_c, best_state, patience = vc, {'t': trunk.state_dict(), 'h': head.state_dict()}, 0
    else:
        patience += 1
    if epoch % 5 == 0:
        with torch.no_grad(): _, vnll = loss_on(va_idx)
        print('epoch %3d  val_C %.4f  val_nll %.4f  (best_C %.4f)' % (epoch, vc, float(vnll), best_c))
    if patience >= 30: print('early stop @', epoch); break
trunk.load_state_dict(best_state['t']); head.load_state_dict(best_state['h'])
print('restored best val C-index =', round(best_c, 4))

In [ ]:
# 7 · evaluate vs the local baseline  (same metrics: C-index + Integrated Brier)
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw, integrated_brier_score
from sklearn.metrics import roc_auc_score

pmf_te, clo_te, pri_te = predict(te_idx)

def time_grid(tt, ee, tr_t, n=20):
    ev = tt[ee.astype(bool)]; lo, hi = np.percentile(ev, 5), np.percentile(ev, 95)
    up = min(tt.max(), tr_t.max()); lw = max(tt.min(), tr_t.min())
    return np.linspace(max(lo, lw + 1e-3), min(hi, up - 1e-3), n)

def surv_at(pmf, times):
    cum = np.cumsum(pmf, 1); right = cuts[1:]; S = np.ones((pmf.shape[0], len(times)))
    for j, t in enumerate(times):
        k = np.searchsorted(right, t, side='right')
        if k > 0: S[:, j] = 1.0 - cum[:, min(k - 1, pmf.shape[1] - 1)]
    return S

tr_t, tr_e = durations[split == 0], events[split == 0]
te_t, te_e = durations[split == 2], events[split == 2]
grid = time_grid(te_t, te_e, tr_t)
y_tr = Surv.from_arrays(tr_e.astype(bool), np.clip(tr_t, 1, None))
y_te = Surv.from_arrays(te_e.astype(bool), np.clip(te_t, 1, None))
risk = risk_from_pmf(pmf_te)                            # expected-bin (outlier-immune)

cH = concordance_index_censored(te_e.astype(bool), te_t, risk)[0]
cI = concordance_index_ipcw(y_tr, y_te, risk, tau=float(grid.max()))[0]
ibs = integrated_brier_score(y_tr, y_te, surv_at(pmf_te, grid), grid)
print('DeepHit v2     :  C-Harrell %.4f   C-IPCW %.4f   IBS %.4f' % (cH, cI, ibs))
print('XGBoost AFT base:  C-Harrell 0.5995   C-IPCW 0.6009   IBS 0.0546   <-- beat C')
cAUC = roc_auc_score(road[te_idx].cpu().numpy(), clo_te)
pmask = prio[te_idx].cpu().numpy() >= 0
pAUC = roc_auc_score(prio[te_idx].cpu().numpy()[pmask], pri_te[pmask])
print('aux heads:  road-closure AUC %.3f   priority AUC %.3f  (priority is ~trivially set by corridor)' % (cAUC, pAUC))

In [ ]:
# 8 · save weights + full predictions, then download into local models/
os.makedirs('out', exist_ok=True)
torch.save({'trunk': trunk.state_dict(), 'head': head.state_dict(), 'cuts': cuts,
            'config': {'d': 64, 'K': int(K), 'cards': cards, 'n_num': int(n_num),
                       'fnode': int(fnode), 'L': int(L)}}, 'out/trunk_deephit.pt')

pmf_all, clo_all, pri_all = predict(torch.arange(N, device=device))
np.savez_compressed('out/preds_all.npz',
    pmf=pmf_all, closure_prob=clo_all, priority_prob=pri_all, cuts=cuts,
    event_id=z['event_id'], split=split, durations=durations, events=events,
    node_id=z['node_id'], road_closure=z['road_closure'], priority_high=z['priority_high'])
print('saved -> out/trunk_deephit.pt , out/preds_all.npz')
try:
    from google.colab import files
    files.download('out/trunk_deephit.pt'); files.download('out/preds_all.npz')
except Exception as e:
    print('(not on Colab; files are in ./out)', e)

## Handoff back to local

Place both downloaded files into the local repo:

```
models/trunk_deephit.pt
models/preds_all.npz
```

Then the next local (CPU) steps run:
- **Conformal calibration** (`calibrate/`) — turns the PMF into *coverage-guaranteed* duration bounds
  (this is what fixes the Integrated Brier / calibration gap).
- **OR-Tools allocator** (`optimize/`) — calibrated severity + closure probability → officer / barricade
  / diversion plan.

> If C-index still trails the baseline, **paste the per-epoch training log** (the `epoch … val_C …`
> lines) back — it tells us whether it under- or over-fit so we can adjust precisely.
